## 03. SQL分析・特徴量エンジニアリング
> パイプライン位置: 01_data_collection → 02_eda → `03_sql` → 04_modeling

---

### 目的
`02_eda.ipynb`で確認したラグ分析の結果を反映し、モデリングに使用するmaster_tableを作成する。

In [1]:
import pandas as pd
import sys
import os

sys.path.append(os.path.abspath('..'))

from src.db.connection import engine

print(pd.read_sql("SELECT * FROM kamis_egg_retail LIMIT 5", engine))
print(pd.read_sql("SELECT * FROM ecos_ppi LIMIT 5", engine))

  year_month  egg_price
0 2016-01-01       5493
1 2016-02-01       5473
2 2016-03-01       5260
3 2016-04-01       5259
4 2016-05-01       5216
  year_month  egg_ppi  electricity_ppi  feed_ppi  egg_cpi
0 2016-01-01    92.17            98.81     95.69   92.594
1 2016-02-01    85.08            98.81     95.69   91.098
2 2016-03-01    83.52            98.81     94.82   87.269
3 2016-04-01    89.81            98.81     94.82   86.773
4 2016-05-01    85.53            98.81     94.82   84.645


- `shift()`でもラグ生成は可能だが、データの整合性と再現性を高めるため    
SQLの`LAG() OVER (ORDER BY year_month)`を使用した。    
DB段階で特徴量を確定することで、以降のモデリングコードが簡潔になり、常に同一の方法で再現できる。


In [2]:
query_join = """
SELECT
    a.year_month,
    a.egg_price,
    b.feed_ppi,
    b.electricity_ppi,
    b.egg_ppi,
    b.egg_cpi
FROM kamis_egg_retail a
JOIN ecos_ppi b ON a.year_month = b.year_month
ORDER BY a.year_month
"""

df_join = pd.read_sql(query_join, engine)

print("JOIN結果 行数:", df_join.shape)  
print(df_join.head())
print(df_join.isnull().sum())

JOIN結果 行数: (120, 6)
  year_month  egg_price  feed_ppi  electricity_ppi  egg_ppi  egg_cpi
0 2016-01-01       5493     95.69            98.81    92.17   92.594
1 2016-02-01       5473     95.69            98.81    85.08   91.098
2 2016-03-01       5260     94.82            98.81    83.52   87.269
3 2016-04-01       5259     94.82            98.81    89.81   86.773
4 2016-05-01       5216     94.82            98.81    85.53   84.645
year_month         0
egg_price          0
feed_ppi           0
electricity_ppi    0
egg_ppi            0
egg_cpi            0
dtype: int64


- 02_edaの時系列チャートで2021年を境に鶏卵価格の水準が構造的に上昇したことが視覚的に確認されたため、    
2021年以降をregime=1とするダミー変数を追加した。数値検証は04_modelingで実施する。

In [3]:
query_lag = """
SELECT
    year_month,
    egg_price,
    feed_ppi,
    electricity_ppi,
    egg_ppi,
    egg_cpi,
    LAG(feed_ppi, 1) OVER (ORDER BY year_month) AS feed_lag1,
    LAG(feed_ppi, 2) OVER (ORDER BY year_month) AS feed_lag2,
    LAG(feed_ppi, 3) OVER (ORDER BY year_month) AS feed_lag3,
    LAG(electricity_ppi, 1) OVER (ORDER BY year_month) AS elec_lag1,
    LAG(electricity_ppi, 2) OVER (ORDER BY year_month) AS elec_lag2,
    LAG(electricity_ppi, 3) OVER (ORDER BY year_month) AS elec_lag3,
    CASE WHEN EXTRACT(YEAR FROM year_month::date) >= 2021 THEN 1 ELSE 0 END AS regime
FROM (
    SELECT
        a.year_month,
        a.egg_price,
        b.feed_ppi,
        b.electricity_ppi,
        b.egg_ppi,
        b.egg_cpi
    FROM kamis_egg_retail a
    JOIN ecos_ppi b ON a.year_month = b.year_month
    ORDER BY a.year_month
)
"""

df_master = pd.read_sql(query_lag, engine)
print("ラグ変数追加後:", df_master.shape) 
print(df_master.head())
print(df_master.isnull().sum())

ラグ変数追加後: (120, 13)
  year_month  egg_price  feed_ppi  electricity_ppi  egg_ppi  egg_cpi  \
0 2016-01-01       5493     95.69            98.81    92.17   92.594   
1 2016-02-01       5473     95.69            98.81    85.08   91.098   
2 2016-03-01       5260     94.82            98.81    83.52   87.269   
3 2016-04-01       5259     94.82            98.81    89.81   86.773   
4 2016-05-01       5216     94.82            98.81    85.53   84.645   

   feed_lag1  feed_lag2  feed_lag3  elec_lag1  elec_lag2  elec_lag3  regime  
0        NaN        NaN        NaN        NaN        NaN        NaN       0  
1      95.69        NaN        NaN      98.81        NaN        NaN       0  
2      95.69      95.69        NaN      98.81      98.81        NaN       0  
3      94.82      95.69      95.69      98.81      98.81      98.81       0  
4      94.82      94.82      95.69      98.81      98.81      98.81       0  
year_month         0
egg_price          0
feed_ppi           0
electricity_ppi  

In [4]:
df_master = df_master.dropna().reset_index(drop=True)

print("欠損値除去後:", len(df_master), "行")  
print(df_master.isnull().sum())  

欠損値除去後: 117 行
year_month         0
egg_price          0
feed_ppi           0
electricity_ppi    0
egg_ppi            0
egg_cpi            0
feed_lag1          0
feed_lag2          0
feed_lag3          0
elec_lag1          0
elec_lag2          0
elec_lag3          0
regime             0
dtype: int64


In [5]:
df_master.to_sql("master_table", engine, if_exists="replace", index=False)

df_master_db = pd.read_sql("SELECT * FROM master_table", engine)

print("格納前 行数:", len(df_master))
print("格納後 行数:", len(df_master_db))
print("一致確認:", len(df_master) == len(df_master_db))
print(df_master_db.isnull().sum())

格納前 行数: 117
格納後 行数: 117
一致確認: True
year_month         0
egg_price          0
feed_ppi           0
electricity_ppi    0
egg_ppi            0
egg_cpi            0
feed_lag1          0
feed_lag2          0
feed_lag3          0
elec_lag1          0
elec_lag2          0
elec_lag3          0
regime             0
dtype: int64


In [6]:
df_master.to_csv("../data/processed/master_table.csv", index=False)
print("マスターテーブル CSV保存完了")

マスターテーブル CSV保存完了
